# Accelerating MRI Acquisition with Deep Learning for Brain and Knee Imaging
### An overview of the project:
stage 1: 2 MT (power=500, offset=2; power=1500, offset=50) and 2 T1 (fa=5 fa=20) predicting 3 MT and 1 T1 \
stage 2: combining the gt and intermediate result from the stage 1, 5MT and 2 T1 are used to predict 5 MT and 1 T1 \

### Indexing of MT and T1
You will probably confused by the indexing in configure file \
Here is how it works while generated:
```python
powers = [500 1500];
offsets = [2 5 10 20 50];
id_flatten = power_id*5+offset_id;
power_id = id_flatten // 5
offset_id = id_flatten % 5
```
So for MT data, ```in: [7]``` means that MT of ```power=1500, offset=10``` is the input data

### How to run:
1. config for stage1
2. run!
3. config for stage 2
4. run again this same jupyter notebook

Prepare configuration file for experiment and data in 'configs' folder

In [ ]:
""" EDIT: Configuration"""
config_name = './configs/OA_stage1_T1=0,2.yaml' # Configuration

# loading and importing
import torch
print("Is CUDA available:", torch.cuda.is_available())
print("PyTorch CUDA version:", torch.version.cuda)
print("CUDA devices available:", torch.cuda.device_count())

import argparse
import scipy.io as sio
import numpy as np
import os
import time
import yaml

# load module
import preprocessing

# Experiment setting
os.environ['CUDA_DEVICE_ORDER'] = 'PCI_BUS_ID'
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
checkpoints_dir = './checkpoints'

# loading settings; Read YAML config
with open(config_name, 'r') as f:
    config = yaml.safe_load(f)
experiment_cfg = config["experiment"]
name = experiment_cfg["experiment_name"]
os.environ['CUDA_VISIBLE_DEVICES'] = experiment_cfg["training_device"]

# prepare workspace for experiment
exp_dir = os.path.join(checkpoints_dir,name)
if not os.path.isdir(exp_dir):
    os.makedirs(exp_dir)
input_data, output_data = preprocessing.load_all_data(config["datasets_train"])
test_input, test_output = preprocessing.load_all_data(config["datasets_test"])

# preparing for training
if experiment_cfg["is_stage2"]:
    pth_name = config["load_stage1_model_path"]
    # idx_substituted = [1,2,3,5,6]
    idx_substituted = [1,2,3,6]
    stage1_input_indx = [0,4,5,7] # [0,4,7] 
    input_data[:,idx_substituted] = preprocessing.stage1_prediction(input_data[:,stage1_input_indx], pth_name,len(stage1_input_indx),len(idx_substituted))
    test_input[:,idx_substituted] = preprocessing.stage1_prediction(test_input[:,stage1_input_indx], pth_name,len(stage1_input_indx),len(idx_substituted))  
train, valid, test = preprocessing.train_test_split(input_data, output_data, test_input, test_output)

# Training
Configure training parameter at configure file, experiment field \
No need to change code here

In [ ]:
# training setup
import os
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import models
# Unpack values into variables
model_name = experiment_cfg["model_name"]
# exp_dir = experiment_cfg["exp_dir"]
batch_size = int(experiment_cfg["batch_size"])
epochs = int(experiment_cfg["epochs"])
learning_rate = float(experiment_cfg["learning_rate"])
in_channel = int(experiment_cfg["in_channel"])
out_channel = int(experiment_cfg["out_channel"])
early_stop_patience = int(experiment_cfg["early_stop_patience"])

import logging
log_path = os.path.join(exp_dir, model_name+".log")
logging.basicConfig(
    filename=log_path,
    filemode='a',  # Append mode
    format='%(asctime)s - %(message)s',
    level=logging.INFO
)

# Log everything
logging.info("Loaded experiment configuration:")
logging.info(f"Model Name: {model_name}")
logging.info(f"Experiment Directory: {exp_dir}")
logging.info(f"Batch Size: {batch_size}")
logging.info(f"Epochs: {epochs}")
logging.info(f"Learning Rate: {learning_rate}")
logging.info(f"In Channels: {in_channel}")
logging.info(f"Out Channels: {out_channel}")
logging.info(f"Early Stop Patience: {early_stop_patience}")
for k, v in experiment_cfg.items():
    print(f"{k}: {v}")

# Setup training device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device: ", device)

# load model
model = models.unet(in_channel,out_channel)
model.to(device)

# preparing model directory
if not os.path.exists(exp_dir):
    os.makedirs(exp_dir)
model_path = os.path.join(exp_dir, model_name)
if os.path.exists(model_path):
    state_dict = torch.load(model_path)
    model.load_state_dict(state_dict)
    model.to(device)

In [ ]:
# Training
X = train['in']
Y = train['out']
X_valid = valid['in']
Y_valid = valid['out']

train_dataset = TensorDataset(X, Y)
val_dataset = TensorDataset(X_valid, Y_valid)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, 1, shuffle=False) # batch_size=batch_size//2

# Initialize model, loss function, optimizer, and scheduler
print("To:", device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.2, patience=25, verbose=True)
# Early Stopping Setup
early_stop_patience = 50
patience_counter = 0

# Training Loop
start_time = time.time()
best_val_loss = float('inf')
model_path = os.path.join(exp_dir, model_name)

for epoch in range(1, epochs + 1):
    model.train()
    train_loss = 0.0

    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device).float(), targets.to(device).float()
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_loss += loss.item() * inputs.size(0)
        # Free memory
        del inputs, targets, outputs
        torch.cuda.empty_cache()
        
    train_loss /= len(train_loader.dataset)

    # Validation Loop
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for inputs, targets in val_loader:
            torch.cuda.empty_cache() 
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            val_loss += loss.item() * inputs.size(0)

    val_loss /= len(val_loader.dataset)

    # Adjust learning rate
    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]['lr']

    # Save the best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0  # Reset counter
        torch.save(model.state_dict(), model_path)
        logging.info(f"Epoch {epoch}: Validation loss improved. Model saved to {model_path}")
    else:
        patience_counter += 1
        logging.info(f"Epoch {epoch}: LR: {current_lr:.2e}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")
        
        if patience_counter >= early_stop_patience:
            logging.info(f"Early stopping at epoch {epoch} due to no improvement for {early_stop_patience} epochs.")
            break


    print(f"Epoch {epoch}/{epochs}, LR: {current_lr:.2e}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")


elapsed_time = (time.time() - start_time) / 60
print(f"Elapsed time: {elapsed_time:.2f} minutes")
logging.info(f"Training completed in {elapsed_time:.2f} minutes.")
logging.info(f"Best validation loss: {best_val_loss:.4f}")

# Testing and visualization
Configure testing parameter at configure file, experiment field \
No need to change code here

In [ ]:
# testing and visualization
import matplotlib.pyplot as plt
import torch
# load module
from utils import display_pytorch
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

train_inorder = {'in': input_data, 'out': output_data}
# data = train_inorder
data = test
N_obj = data['in'].shape[0] # CHANGE: train test
num_test_samples = N_obj
batch_size = 2 # Try a smaller batch size
num_batches = (num_test_samples + batch_size - 1) // batch_size # Ceiling division

all_predictions_cpu = []
model.eval() # Ensure model is in evaluation mode
with torch.no_grad():
    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = min((i + 1) * batch_size, num_test_samples)

        input_batch_cpu = data['in'][start_idx:end_idx] # CHANGE: train test
        input_batch_gpu = input_batch_cpu.to(device)

        predictions_batch_gpu = model(input_batch_gpu)
        predictions_batch_cpu = predictions_batch_gpu.cpu()
        all_predictions_cpu.append(predictions_batch_cpu)

# Concatenate predictions if needed
final_predictions_cpu = torch.cat(all_predictions_cpu, dim=0)
print(f"Final predictions shape: {final_predictions_cpu.shape}")

# Number of input and output channels
num_input_channels = in_channel
num_output_channels = out_channel

# config files and visualzation
save_dir = experiment_cfg["visualization_dir"] + model_name + '/'
start_idx, end_idx = 0, 10
# visulization
os.makedirs(save_dir, exist_ok=True)
for slice_idx in range(start_idx, end_idx):
    display_pytorch(data['out'][slice_idx],final_predictions_cpu[slice_idx],os.path.join(save_dir,'{}.png'.format(slice_idx+1)))
print("visualzization saved to",save_dir)

# Correlation analysis
No need to change code here \
Only run it on stage 2; if want to run stage2, go to config and change the experiment:is_stage2 param

In [ ]:
# Correlation analysis
from math import log10 # For PSNR calculation, ensure direct log10
MAX_PIXEL_VALUE = 1.0

num_groups = 4
group_size = 90

mse_sum = [0,0,0,0]

total_channel = 10

for group_idx in range(num_groups):
    print(f"\n--- Metrics for Group {group_idx + 1} ---")
    
    # Define the slice for the current group
    start_idx = group_idx * group_size
    end_idx = (group_idx + 1) * group_size

    # Extract ground truth and predictions for the current group
    group_true_images = torch.cat((test_input[start_idx:end_idx,idx_substituted], test['out'][start_idx:end_idx]), dim=1)[:, [0,1,2,4,5,6,7,8,3,9]] # CHANGE: train test
    group_pred_images = torch.cat((test['in'][start_idx:end_idx, idx_substituted],final_predictions_cpu[start_idx:end_idx]), dim=1)[:, [0,1,2,4,5,6,7,8,3,9]]


    print(f"  Images in this group: {group_true_images.shape[0]}")
    print("\n  Channel |      MAE     |      MSE     |    PSNR (dB)")
    print("  --------|--------------|--------------|--------------")

    # Iterate through each channel within the current group's images
    for channel_idx in range(total_channel):
        # Extract data for the specific channel across all images in the group
        # This gives a tensor of shape (group_size, H, W)
        true_channel_data = group_true_images[:, channel_idx, :, :]
        pred_channel_data = group_pred_images[:, channel_idx, :, :]

        # Calculate absolute difference and squared difference for all pixels in this channel, across all images
        abs_diff = torch.abs(true_channel_data - pred_channel_data)
        squared_diff = (true_channel_data - pred_channel_data)**2

        # Average the MAE and MSE across all pixels and all images in the group for this channel
        channel_mae = torch.mean(abs_diff).item()
        channel_mse = torch.mean(squared_diff).item()
        mse_sum[group_idx] += channel_mse / total_channel

        # Calculate PSNR for the current channel
        if channel_mse == 0:
            # If MSE is 0, images are identical for this channel, PSNR is infinite
            channel_psnr = float('inf')
        else:
            # Add a small epsilon to MSE if it's very close to zero to prevent log(0) issues,
            # though an MSE of 0 is a perfect match.
            # Using log10 from math module for clarity and directness
            channel_psnr = 10 * log10((MAX_PIXEL_VALUE**2) / channel_mse)

        # Print the results for the current channel, formatted nicely
        print(f"  {channel_idx:<7d} | {channel_mae:<12.6f} | {channel_mse:<12.6f} | {channel_psnr:<12.2f}")
        

print("\n" + "="*100) # Final separator
print("Image similarity metric calculation complete.")
print(mse_sum)

In [ ]:
# Correlation fitting
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
import torch

# Assuming `save_dir`, `num_groups`, `group_size`, `test_input`, `test`,
# `final_predictions_cpu`, `idx_substituted`, `num_channels`, and `out_channel` are already defined.
# For example:
# save_dir = "./correlation/"
# num_correlation_samples = 10
num_channels = 10
# out_channel = 10
# num_groups = 1
# group_size = 50
# test_input = torch.randn(50, 10, 256, 256)
# test = {'out': torch.randn(50, 10, 256, 256), 'in': torch.randn(50, 10, 256, 256)}
# final_predictions_cpu = torch.randn(50, 10, 256, 256)
# idx_substituted = torch.arange(10)

# The following code is modified to sample from all images in a group.

for group_idx in range(num_groups):
    print(f"\n--- Correlation Analysis for Group {group_idx + 1} ---")
    
    # Define the slice for the current group
    start_idx = group_idx * group_size
    end_idx = (group_idx + 1) * group_size

    # Extract ground truth and predictions for the entire group
    group_true_images = torch.cat((test_input[start_idx:end_idx, idx_substituted], test['out'][start_idx:end_idx]), dim=1)[:, [0, 1, 2, 4, 5, 6, 7, 8, 3, 9]]
    group_pred_images = torch.cat((test['in'][start_idx:end_idx, idx_substituted], final_predictions_cpu[start_idx:end_idx]), dim=1)[:, [0, 1, 2, 4, 5, 6, 7, 8, 3, 9]]
    
    # Reshape the entire group's images for plotting: (N*H*W, C)
    # This flattens all images in the group into one large collection of pixels
    ground_truth_for_plot = group_true_images.permute(0, 2, 3, 1).reshape(-1, num_channels).numpy()
    predicted_for_plot = group_pred_images.permute(0, 2, 3, 1).reshape(-1, num_channels).numpy()

    # Determine a suitable number of pixels to display for scatter plot
    num_pixels = ground_truth_for_plot.shape[0]  # Total number of pixels in the group
    display_size = min(2000, num_pixels)
    display_idx = np.random.choice(num_pixels, size=display_size, replace=False)

    # Create subplots for each channel
    rows = 2
    fig, axes = plt.subplots(rows, 5, figsize=(25, 5 * rows))
    fig.suptitle(f'Correlation for Group {group_idx + 1}', fontsize=16)
    
    # Flatten axes for easy iteration
    if axes.ndim == 1:
        axes = axes.reshape(1, -1)
    axes = axes.flatten()

    for i in range(num_channels):
        ax = axes[i]
        
        # Extract pixel values for the current channel using the random indices
        x = ground_truth_for_plot[display_idx, i]
        y = predicted_for_plot[display_idx, i]
        
        # Create scatter plot
        ax.scatter(x, y, alpha=0.5, s=1)
        
        if len(x) > 1:
            slope, intercept = np.polyfit(x, y, 1)
            r2 = r2_score(x, y)
            
            print(f"  Group {group_idx + 1}, Channel {i + 1}: y = {slope:.6f}x + {intercept:.6f}, R^2 = {r2:.6f}")
            
            x_min, x_max = np.min(x), np.max(x)
            if x_max > x_min:
                plot_x = np.array([x_min, x_max])
                ax.plot(plot_x, slope * plot_x + intercept, color='gray', linestyle='--', linewidth=1)
        else:
            print(f"  Group {group_idx + 1}, Channel {i + 1}: Not enough data points to calculate regression stats.")

        ax.set_xlabel('Ground Truth', fontsize=12)
        ax.set_ylabel(f'Predicted Channel {i + 1}', fontsize=12)
        ax.grid(True)
        ax.tick_params(axis='both', which='major', labelsize=10)

    # Hide any unused subplots
    for i in range(num_channels, rows * 3):
        if i < len(axes):
            fig.delaxes(axes[i])

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.savefig(f"{save_dir}correlation_group_{group_idx + 1}.png")
    plt.show()
    plt.close(fig)

# export to mat
This part is hardcoded. Consider editing it according to your data

In [ ]:
"""
Edit: oa_id stores the id order of the imported mat file data. Most of the time just copy from the matlab file used for generating testing data
"""
oa_ids = [53, 58, 69,   7,  55 ,62   ,37, 43, 47  , 14 ,21, 22]

# laod result to mat file
import scipy.io
import numpy as np

MT_in_index = config["datasets_train"][0]["in"]
MT_out_index = config["datasets_train"][0]["out"]
T1_in_index = config["datasets_train"][1]["in"]
T1_out_index = config["datasets_train"][1]["out"]

start_ind = 0

for oa_id in oa_ids:
    end_ind = start_ind + 30
    # MT
    MTimg = np.zeros((256, 256, 30, 10))
    img_in = test["in"][start_ind:end_ind,:5]
    img_out = final_predictions_cpu[start_ind:end_ind,:5]
    MTimg[:,:,:,MT_in_index] = img_in.permute(2,3,0,1).float()
    MTimg[:,:,:,MT_out_index] = img_out.permute(2,3,0,1).float()
    mat_dict = {
            'MTimg': MTimg,
        }
    scipy.io.savemat(f'OA_test/OA_{oa_id}_MT_predicted.mat', mat_dict)
    
    # T1
    T1img = np.zeros((256, 256, 30, 4))
    img_in = test["in"][start_ind:end_ind,5:]
    img_out = final_predictions_cpu[start_ind:end_ind,5:]
    T1img[:,:,:,T1_in_index] = img_in.permute(2,3,0,1).float()
    T1img[:,:,:,T1_out_index] = img_out.permute(2,3,0,1).float()
    mat_dict = {
            'T1img': T1img,
        }
    scipy.io.savemat(f'OA_test/OA_{oa_id}_T1_predicted.mat', mat_dict)

    start_ind += 30